# 02 — Pilot runner
Start with **one task × three conditions**. Do not jump straight to 60 paid runs.

In [ ]:
from pathlib import Path
import os, sys, json, yaml, pandas as pd
ROOT=Path.cwd(); ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT
sys.path.insert(0,str(ROOT))
from dotenv import load_dotenv
load_dotenv(ROOT/'.env')
from src.experiment import load_config, load_instances, run_one, export_predictions
cfg=load_config()
print(cfg['pilot'])

In [ ]:
instances=load_instances(cfg)
[(x['instance_id'],x.get('repo')) for x in instances]

### One-task smoke run
This **will call Azure and start a SWE-bench Docker task container**.

In [ ]:
task=instances[0]
records=[]
# Uncomment only after notebook 00 + 01 pass.
# for condition in cfg['pilot']['conditions']:
#     print('RUN',task['instance_id'],condition)
#     records.append(run_one(task,condition,cfg))
# pd.DataFrame(records)[['instance_id','condition','exit_status','elapsed_seconds','error']]

### Export predictions
Official grading must be run separately for each condition/run ID so result caching cannot mix patches.

In [ ]:
# Example after records exist:
# for condition in cfg['pilot']['conditions']:
#     subset=[r for r in records if r['condition']==condition]
#     export_predictions(subset, ROOT/'results'/f'predictions_{condition}.jsonl')

### Official SWE-bench evaluation
Use unique `run_id` values for every condition. Start with `max_workers=1`. The evaluation harness applies the patch in Docker and produces the resolved/unresolved result; do not use the agent's self-reported success as the outcome.

In [ ]:
from src.evaluate import swebench_eval_command
for condition in cfg['pilot']['conditions']:
    p=ROOT/'results'/f'predictions_{condition}.jsonl'
    print(' '.join(swebench_eval_command(cfg['pilot']['dataset_name'],p,f'pilot_{condition}',max_workers=1)))

After one clean task, raise `pilot.max_tasks` to 3 for a smoke pilot. Only after that should you predeclare ~20 held-out tasks and run all three conditions.